In [ ]:
import os
import requests
import xml.etree.ElementTree as ET
import datawrapper as dw


API_URL = "https://data.hub.api.metoffice.gov.uk/nswws/v1.1/objects/"
chart_id = "kNKqu"

def get_live_warnings():
    api_key = os.environ.get("MET_OFFICE_API_KEY")
    if not api_key:
        print("Missing MET_OFFICE_API_KEY environment variable.")
        return []

    headers = {
        "ApiKey": api_key,
        "Accept": "application/atom+xml"
    }

    try:
        print(f"Fetching live warnings feed from {API_URL}...")
        FEED_URL = API_URL + "feed/"
        response = requests.get(FEED_URL, headers=headers)
        
        if response.status_code == 200:
            data = ET.fromstring(response.content)
            namespace = {"atom": "http://www.w3.org/2005/Atom"}
            entry_id = data.findtext("atom:id", namespaces=namespace)
            if not entry_id or not entry_id.strip():
                print("No UUID found in the feed response.")
                return []

            uuid = entry_id.strip()
            if uuid.lower().startswith("urn:uuid:"):
                uuid = uuid[len("urn:uuid:"):]

            WARNING_URL = API_URL + "issued/" + uuid
            response = requests.get(WARNING_URL, headers=headers)
            data = response.json()
            geometry = data["features"][0]["geometry"]

            chart = dw.get_chart(chart_id)
            chart.data = geometry
            chart.update()

        else:
            print(f"Failed to fetch warning data. Status code: {response.status_code}")
            print(response.text)

        
                    
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

if __name__ == "__main__":
    get_live_warnings()
